# Metadata Filtering And Time Aware Search

**Notebook:** `06-metadata-filtering-and-time-aware-search.ipynb`

This notebook is part of the Personal Knowledge Engine project.

In [1]:
# Start coding here
# ==========================================================
# Notebook 06: Metadata Filtering & Time-Aware Search
# ==========================================================

import re
import faiss
import pickle
import numpy as np
import pandas as pd

from datetime import datetime
from datetime import timedelta

from sentence_transformers import SentenceTransformer

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
chunks_df = pd.read_csv("data/processed/chunked_corpus.csv")

chunks_df.head()

,chunk_id,source,category,created_date,chunk_text
0,1,2026-06-14.md,journal,2026-06-14,# Daily Journal\n\nToday I read about local AI...
1,2,ai_resume_matcher.md,project,NaN,# AI Resume Matcher\n\nToday I worked on the F...
2,3,book.pdf,book,NaN,Praise for Natural Language Processing with Tr...
3,4,book.pdf,book,NaN,NLP than the creators of said library? Natura...
4,5,book.pdf,book,NaN,insight and deftly mixes research advances wit...


In [3]:
with open("data/embeddings/chunk_embeddings.pkl", "rb") as file:

    embeddings = pickle.load(file)

embeddings = np.array(embeddings).astype("float32")

In [4]:
index = faiss.read_index("data/embeddings/faiss_index.bin")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 18bbf38a-0291-408b-af9f-ddc9cff395cf)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
project_notes = chunks_df[chunks_df["category"] == "project"]

project_notes.head()

,chunk_id,source,category,created_date,chunk_text
1,2,ai_resume_matcher.md,project,NaN,# AI Resume Matcher\n\nToday I worked on the F...


In [6]:
journal_notes = chunks_df[chunks_df["category"] == "journal"]

journal_notes.head()

,chunk_id,source,category,created_date,chunk_text
0,1,2026-06-14.md,journal,2026-06-14,# Daily Journal\n\nToday I read about local AI...


In [7]:
journal_notes = chunks_df[chunks_df["category"] == "journal"]

journal_notes.head()

,chunk_id,source,category,created_date,chunk_text
0,1,2026-06-14.md,journal,2026-06-14,# Daily Journal\n\nToday I read about local AI...


In [8]:
chunks_df.dtypes

chunk_id         int64
source          object
category        object
created_date    object
chunk_text      object
dtype: object

In [9]:
def resolve_time_expression(query):

    today = datetime.today()

    query = query.lower()

    if "yesterday" in query:

        return today - timedelta(days=1)

    if "last week" in query:

        return today - timedelta(days=7)

    if "last month" in query:

        return today - timedelta(days=30)

    if "past 30 days" in query:

        return today - timedelta(days=30)

    return None

In [10]:
resolve_time_expression("show me ai projects from last week")

datetime.datetime(2026, 6, 7, 10, 48, 29, 842396)

In [11]:
CATEGORIES = ["project", "journal", "book", "general"]

In [12]:
def extract_category(query):

    query = query.lower()

    for category in CATEGORIES:

        if category in query:

            return category

    return None

In [13]:
extract_category("show me project notes about faiss")

'project'

In [14]:
def semantic_search(query, top_k=20):

    query_embedding = model.encode(query)

    query_embedding = np.array([query_embedding]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = chunks_df.iloc[indices[0]].copy()

    results["score"] = scores[0]

    return results

In [15]:
def apply_metadata_filters(dataframe, category=None, start_date=None):

    filtered = dataframe.copy()

    if category:

        filtered = filtered[filtered["category"] == category]

    if start_date:

        filtered = filtered[filtered["created_date"] >= start_date]

    return filtered

In [16]:
def metadata_aware_search(query, top_k=10):

    # -------------------------
    # Extract Metadata
    # -------------------------

    category = extract_category(query)

    start_date = resolve_time_expression(query)

    # -------------------------
    # Semantic Search
    # -------------------------

    results = semantic_search(query, top_k=50)

    # -------------------------
    # Metadata Filtering
    # -------------------------

    results = apply_metadata_filters(results, category=category, start_date=start_date)

    return results.head(top_k)

In [17]:
results = metadata_aware_search("show me project notes about faiss")

results[["source", "category", "score", "chunk_text"]]

,source,category,score,chunk_text
1,ai_resume_matcher.md,project,0.33321,# AI Resume Matcher\n\nToday I worked on the F...


In [18]:
results = metadata_aware_search("show me journal entries about local ai")

results[["source", "category", "score"]]

,source,category,score
0,2026-06-14.md,journal,0.601339


In [19]:
results = metadata_aware_search("show me ai projects from last week")

results[["source", "created_date", "score"]]

,source,created_date,score


In [20]:
{
    "tags": ["ai", "rag"],
    "author": "vinay",
    "priority": "high",
    "source_type": "journal",
    "created_date": "2026-06-12",
}

{'tags': ['ai', 'rag'],
 'author': 'vinay',
 'priority': 'high',
 'source_type': 'journal',
 'created_date': '2026-06-12'}

In [21]:
metadata_aware_search("show me ai projects from last week")

,chunk_id,source,category,created_date,chunk_text,score
